# Nuclear Proteome IDR Analysis
This notebook fetches all reviewed human nuclear proteins from UniProt and extracts their Intrinsically Disordered Regions (IDRs) using metapredict.

**Just click Runtime → Run all** to execute everything.

In [ ]:
!pip install -q metapredict requests

In [ ]:
import requests
import os
import csv
import time
import metapredict as meta

OUTPUT_DIR = "nuclear_proteome_idrs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Query UniProt for reviewed human nuclear proteins
UNIPROT_QUERY = "(organism_id:9606) AND (reviewed:true) AND (cc_scl_term:SL-0191)"
SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"

print("Fetching nuclear proteome from UniProt...")

all_fasta = ""
url = SEARCH_URL
params = {"query": UNIPROT_QUERY, "format": "fasta", "size": 500}
page = 1

while url:
    print(f"  Page {page}...")
    for attempt in range(4):
        try:
            resp = requests.get(url, params=params, timeout=120)
            resp.raise_for_status()
            break
        except requests.RequestException as e:
            time.sleep(2 ** (attempt + 1))
            if attempt == 3: raise
    all_fasta += resp.text
    link = resp.headers.get("Link", "")
    if 'rel="next"' in link:
        url = link.split(";")[0].strip("<>")
        params = {}
    else:
        url = None
    page += 1

# Parse FASTA
proteins = {}
current_id = current_header = None
current_seq = []
for line in all_fasta.strip().split("\n"):
    if line.startswith(">"):
        if current_id:
            proteins[current_id] = {"header": current_header, "sequence": "".join(current_seq)}
        parts = line.split("|")
        current_id = parts[1] if len(parts) >= 2 else line[1:].split()[0]
        current_header = line
        current_seq = []
    else:
        current_seq.append(line.strip())
if current_id:
    proteins[current_id] = {"header": current_header, "sequence": "".join(current_seq)}

# Save raw FASTA
with open(f"{OUTPUT_DIR}/human_nuclear_proteome.fasta", "w") as f:
    f.write(all_fasta)

print(f"\nDownloaded {len(proteins)} human nuclear proteins")

In [ ]:
# Predict IDRs using metapredict
print("Predicting IDRs with metapredict (this may take 10-30 min)...")

idr_results = []
idr_fasta_entries = []
processed = errors = 0

for uniprot_id, data in proteins.items():
    seq = data["sequence"]
    if not seq:
        continue
    try:
        idrs = meta.predict_disorder_domains(seq)
        for i, idr_seq in enumerate(idrs.disordered_domains):
            start = seq.find(idr_seq)
            end = start + len(idr_seq) if start >= 0 else -1
            idr_results.append({
                "uniprot_id": uniprot_id,
                "protein_length": len(seq),
                "idr_index": i + 1,
                "idr_start": start + 1 if start >= 0 else "N/A",
                "idr_end": end if start >= 0 else "N/A",
                "idr_length": len(idr_seq),
                "idr_sequence": idr_seq
            })
            idr_fasta_entries.append(f">{uniprot_id}_IDR{i+1} start={start+1} end={end} len={len(idr_seq)}\n{idr_seq}")
    except Exception as e:
        errors += 1
        if errors <= 3: print(f"  Error: {uniprot_id}: {e}")
    processed += 1
    if processed % 500 == 0:
        print(f"  {processed}/{len(proteins)} proteins processed...")

print(f"\nDone! {processed} proteins, {len(idr_results)} IDR regions, {errors} errors")

In [ ]:
# Save results
with open(f"{OUTPUT_DIR}/nuclear_proteome_idrs.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["uniprot_id","protein_length","idr_index","idr_start","idr_end","idr_length","idr_sequence"], delimiter="\t")
    w.writeheader()
    w.writerows(idr_results)

with open(f"{OUTPUT_DIR}/nuclear_proteome_idrs.fasta", "w") as f:
    f.write("\n".join(idr_fasta_entries))

total_with_idrs = len(set(r["uniprot_id"] for r in idr_results))
avg_len = sum(r["idr_length"] for r in idr_results) / len(idr_results) if idr_results else 0

print(f"Nuclear proteins: {len(proteins)}")
print(f"Proteins with IDRs: {total_with_idrs}")
print(f"Total IDR regions: {len(idr_results)}")
print(f"Avg IDR length: {avg_len:.1f} residues")
print(f"\nFiles saved to {OUTPUT_DIR}/")

In [ ]:
# Download files (click the links that appear)
from google.colab import files
import shutil
shutil.make_archive("nuclear_proteome_idrs", "zip", ".", OUTPUT_DIR)
files.download("nuclear_proteome_idrs.zip")
print("Download started! Check your browser downloads.")